In [9]:
import pandas as pd

input_path  = r"C:\Users\semwi\FPL-Core-Insights\data\match_data.csv"
output_path = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"

cols = [
    "match_id", "season", "round", "timestamp", "status",
    "home_team", "away_team", "home_team_id", "away_team_id",
    "home_goals", "away_goals"
]

df = pd.read_csv(input_path, low_memory=False)
training_data = df[cols].copy()
training_data.to_csv(output_path, index=False)
print(f"training_data.csv aangemaakt met {len(training_data)} rijen en {len(cols)} kolommen.")

training_data.csv aangemaakt met 2665 rijen en 11 kolommen.


In [10]:
import pandas as pd

training_path = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"
elo_path      = r"C:\Users\semwi\FPL-Core-Insights\data\clubelo_history.csv"

training = pd.read_csv(training_path)
elo      = pd.read_csv(elo_path, parse_dates=['elo_from', 'elo_to'])

name_map = {
    'Brighton & Hove Albion':  'Brighton',
    'Ipswich Town':            'Ipswich',
    'Leeds United':            'Leeds',
    'Leicester City':          'Leicester',
    'Luton Town':              'Luton',
    'Manchester City':         'Man City',
    'Manchester United':       'Man Utd',
    'Newcastle United':        'Newcastle',
    'Norwich City':            'Norwich',
    'Nottingham Forest':       "Nott'm Forest",
    'Sheffield United':        'Sheffield Utd',
    'Tottenham Hotspur':       'Spurs',
    'West Bromwich Albion':    'West Brom',
    'West Ham United':         'West Ham',
    'Wolverhampton':           'Wolves',
    'Wolverhampton Wanderers': 'Wolves',
}

training['home_team_elo'] = training['home_team'].replace(name_map)
training['away_team_elo'] = training['away_team'].replace(name_map)
training['match_date']    = pd.to_datetime(training['timestamp'], format='mixed').dt.normalize()

def get_elo(team_name, match_date, elo_df):
    mask = (
        (elo_df['team_name'] == team_name) &
        (elo_df['elo_from']  <= match_date) &
        (elo_df['elo_to']    >= match_date)
    )
    rows = elo_df[mask]
    return rows.iloc[-1]['clubelo_rating'] if not rows.empty else None

print("Home ELO koppelen...")
training['home_elo'] = training.apply(lambda r: get_elo(r['home_team_elo'], r['match_date'], elo), axis=1)
print("Away ELO koppelen...")
training['away_elo'] = training.apply(lambda r: get_elo(r['away_team_elo'], r['match_date'], elo), axis=1)
training['elo_diff'] = training['home_elo'] - training['away_elo']

training = training.drop(columns=['home_team_elo', 'away_team_elo', 'match_date'])

missing = training['home_elo'].isna().sum()
print(f"Klaar! {len(training)} rijen | {missing} zonder ELO")
training.to_csv(training_path, index=False)
print(f"✅ Opgeslagen → {training_path}")

Home ELO koppelen...
Away ELO koppelen...
Klaar! 2665 rijen | 0 zonder ELO
✅ Opgeslagen → C:\Users\semwi\FPL-Core-Insights\data\training_data.csv


In [12]:
"""
Player Lineup Strength
======================
- Berekent per wedstrijd de gemiddelde rating van de basiself per team
- Geen data leakage: elke speler rating is gebaseerd op wedstrijden VOOR die wedstrijd
- Voor toekomstige wedstrijden: gebruikt historische gemiddelde rating van verwachte spelers
- Schrijft home_lineup_strength en away_lineup_strength naar training_data.csv
"""

import pandas as pd

# ── Paden ─────────────────────────────────────────────────────────────────────
PLAYER_DIR    = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\players"
TRAINING_PATH = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"
OUTPUT_PATH   = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"

# ── Data laden ────────────────────────────────────────────────────────────────
import os, glob

print("📂 Player data laden...")
all_files  = sorted(glob.glob(os.path.join(PLAYER_DIR, "*_players.csv")))
players_df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in all_files],
    ignore_index=True
)
print(f"   {len(players_df):,} rijen geladen uit {len(all_files)} bestanden")

training = pd.read_csv(TRAINING_PATH)
print(f"   {len(training):,} wedstrijden in training_data")

# ── Voorbereiding ─────────────────────────────────────────────────────────────

# Zorg dat timestamp numeriek is voor vergelijking
players_df["timestamp"] = pd.to_numeric(players_df["timestamp"], errors="coerce")
training["timestamp"]   = pd.to_numeric(training["timestamp"],   errors="coerce")

# Alleen basiself 
starters = players_df[
    players_df["substitute"].astype(str).str.lower().isin(["false", "expected"])
].copy()

print(f"   {len(starters):,} starter-rijen met rating")

# ── Per speler: rolling gemiddelde rating (alleen vorige wedstrijden) ─────────
print("\n⚙️  Historische ratings per speler berekenen...")

starters = starters.sort_values(["player_id", "timestamp"])

# Shift(1) = gebruik alleen wat je al wist VOOR deze wedstrijd
starters["rating_historic"] = (
    starters.groupby("player_id")["rating"]
    .transform(lambda x: x.expanding().mean().shift(1))
)

# Voor spelers met geen history: gebruik hun overall gemiddelde (voor toekomstige wedstrijden)
global_avg = starters.groupby("player_id")["rating"].mean().rename("rating_global_avg")
starters   = starters.merge(global_avg, on="player_id", how="left")

# Beste schatting: historisch als beschikbaar, anders global avg
starters["rating_best"] = starters["rating_historic"].fillna(starters["rating_global_avg"])

print(f"   Rating beschikbaar voor {starters['rating_best'].notna().sum():,} / {len(starters):,} rijen")

# ── Per wedstrijd: gemiddelde sterkte van de basiself ────────────────────────
print("\n⚙️  Lineup sterkte per wedstrijd berekenen...")

lineup_strength = (
    starters[starters["rating_best"].notna()]
    .groupby(["match_id", "side"])["rating_best"]
    .mean()
    .reset_index()
    .rename(columns={"rating_best": "lineup_strength"})
)

# Split home en away
home_strength = (
    lineup_strength[lineup_strength["side"] == "home"]
    [["match_id", "lineup_strength"]]
    .rename(columns={"lineup_strength": "home_lineup_strength"})
)
away_strength = (
    lineup_strength[lineup_strength["side"] == "away"]
    [["match_id", "lineup_strength"]]
    .rename(columns={"lineup_strength": "away_lineup_strength"})
)

# ── Merge in training_data ────────────────────────────────────────────────────

# Verwijder oude kolommen als die er al in zitten
for col in ["home_lineup_strength", "away_lineup_strength", "lineup_strength_diff"]:
    if col in training.columns:
        training = training.drop(columns=[col])

training = training.merge(home_strength, on="match_id", how="left")
training = training.merge(away_strength, on="match_id", how="left")
training["lineup_strength_diff"] = (
    training["home_lineup_strength"] - training["away_lineup_strength"]
)

# ── Resultaat ─────────────────────────────────────────────────────────────────
filled   = training["home_lineup_strength"].notna().sum()
upcoming = (training["status"] == "Not started").sum()
played   = (training["status"] != "Not started").sum()

print(f"\n   Gespeelde wedstrijden:   {played:,}")
print(f"   Toekomstige wedstrijden: {upcoming:,}")
print(f"   Lineup strength gevuld:  {filled:,} ({filled/len(training)*100:.0f}%)")
print(f"\n   Voorbeeld:")
print(training[["home_team","away_team","home_lineup_strength",
                "away_lineup_strength","lineup_strength_diff"]]
      .dropna(subset=["home_lineup_strength"])
      .tail(5).to_string(index=False))

training.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Opgeslagen → {OUTPUT_PATH}")

📂 Player data laden...
   103,260 rijen geladen uit 7 bestanden
   2,665 wedstrijden in training_data
   58,850 starter-rijen met rating

⚙️  Historische ratings per speler berekenen...
   Rating beschikbaar voor 58,850 / 58,850 rijen

⚙️  Lineup sterkte per wedstrijd berekenen...

   Gespeelde wedstrijden:   2,565
   Toekomstige wedstrijden: 100
   Lineup strength gevuld:  2,665 (100%)

   Voorbeeld:
             home_team         away_team  home_lineup_strength  away_lineup_strength  lineup_strength_diff
             Liverpool         Brentford              7.106681              6.947140              0.159541
Brighton & Hove Albion Manchester United              6.976985              7.050702             -0.073716
       Manchester City       Aston Villa              7.136975              6.947850              0.189124
        Crystal Palace           Arsenal              6.913059              7.043811             -0.130752
               Burnley     Wolverhampton              6.7535

In [15]:
import pandas as pd
import math

# ── Paden ────────────────────────────────────────────────────────────────────
match_path    = r"C:\Users\semwi\FPL-Core-Insights\data\match_data.csv"
training_path = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"
output_path   = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"

# ── Pi Rating parameters (Constantinou & Fenton 2013) ────────────────────────
LAMBDA_MAIN    = 0.05    # leersnelheid
LAMBDA_CROSS   = 0.7   # cross-context catch-up
HOME_ADVANTAGE = 0.2    # H in de formule, aan te passen

# ── Pi Rating klasse ─────────────────────────────────────────────────────────
class PiRatingSystem:
    def __init__(self):
        self.home_ratings = {}
        self.away_ratings = {}

    def _get(self, team):
        return self.home_ratings.get(team, 0.0), self.away_ratings.get(team, 0.0)

    @staticmethod
    def _squash(error, scale=3.0):
        return math.tanh(error / scale) * scale

    def expected_goal_diff(self, home_team, away_team):
        h_home, _ = self._get(home_team)
        _, a_away = self._get(away_team)
        return h_home - a_away + HOME_ADVANTAGE

    def update(self, home_team, away_team, actual_goal_diff):
        h_home, h_away = self._get(home_team)
        a_home, a_away = self._get(away_team)

        expected = h_home - a_away + HOME_ADVANTAGE
        error    = actual_goal_diff - expected
        se       = self._squash(error)

        # Thuisploeg home rating stijgt als beter dan verwacht
        new_h_home = h_home + LAMBDA_MAIN * se
        # Uitploeg away rating daalt als thuisploeg beter dan verwacht
        new_a_away = a_away - LAMBDA_MAIN * se

        # Cross-update: away rating thuisploeg volgt zijn home rating
        new_h_away = h_away + LAMBDA_CROSS * (new_h_home - h_away)
        # Cross-update: home rating uitploeg volgt zijn away rating
        new_a_home = a_home + LAMBDA_CROSS * (new_a_away - a_away)

        self.home_ratings[home_team] = new_h_home
        self.away_ratings[home_team] = new_h_away
        self.home_ratings[away_team] = new_a_home
        self.away_ratings[away_team] = new_a_away

    def season_reset(self):
        """Schaal alle ratings richting 0 aan het einde van een seizoen."""
        for team in self.home_ratings:
            self.home_ratings[team] *= SEASON_DECAY
        for team in self.away_ratings:
            self.away_ratings[team] *= SEASON_DECAY


# ── Data inladen & sorteren op datum ─────────────────────────────────────────
df = pd.read_csv(match_path)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')
df = df.sort_values('timestamp').reset_index(drop=True)

# Splits gespeeld en upcoming
played   = df.dropna(subset=['home_goals', 'away_goals']).copy()
upcoming = df[df['home_goals'].isna()].copy()

# ── Pi ratings berekenen op gespeelde wedstrijden ─────────────────────────────
pi = PiRatingSystem()
records = []

for _, row in played.iterrows():
    home = row['home_team']
    away = row['away_team']
    gd   = row['home_goals'] - row['away_goals']

    h_home_pre, _  = pi._get(home)
    _, a_away_pre  = pi._get(away)
    exp_gd_pre     = pi.expected_goal_diff(home, away)

    pi.update(home, away, gd)

    records.append({
        'match_id'           : row['match_id'],
        'home_pi_home_pre'   : round(h_home_pre, 5),
        'away_pi_away_pre'   : round(a_away_pre, 5),
        'pi_expected_gd_pre' : round(exp_gd_pre,  5),
    })

# ── Upcoming wedstrijden: huidige ratings als pre ─────────────────────────────
for _, row in upcoming.iterrows():
    home = row['home_team']
    away = row['away_team']

    h_home_pre, _  = pi._get(home)
    _, a_away_pre  = pi._get(away)
    exp_gd_pre     = pi.expected_goal_diff(home, away)

    records.append({
        'match_id'           : row['match_id'],
        'home_pi_home_pre'   : round(h_home_pre, 5),
        'away_pi_away_pre'   : round(a_away_pre, 5),
        'pi_expected_gd_pre' : round(exp_gd_pre,  5),
    })

pi_df = pd.DataFrame(records)  # ← deze regel toevoegen

# ── Validatie prints ──────────────────────────────────────────────────────────
print("\n=== Rating bereik ===")
all_home = list(pi.home_ratings.values())
all_away = list(pi.away_ratings.values())
print(f"Home ratings: min={min(all_home):.3f}, max={max(all_home):.3f}, gemiddeld={sum(all_home)/len(all_home):.3f}")
print(f"Away ratings: min={min(all_away):.3f}, max={max(all_away):.3f}, gemiddeld={sum(all_away)/len(all_away):.3f}")

print("\n=== Top 5 sterkste teams (home) ===")
sorted_home = sorted(pi.home_ratings.items(), key=lambda x: x[1], reverse=True)
for team, rating in sorted_home[:5]:
    print(f"  {team}: {rating:.4f}")

print("\n=== Top 5 zwakste teams (home) ===")
for team, rating in sorted_home[-5:]:
    print(f"  {team}: {rating:.4f}")

print("\n=== Verwacht doelsaldo (pi_expected_gd_pre) ===")
print(pi_df['pi_expected_gd_pre'].describe())

# ── Samenvoegen met training_data ─────────────────────────────────────────────
training = pd.read_csv(training_path)
training = training[[c for c in training.columns if 'pi' not in c]]
training['match_id'] = training['match_id'].astype(float)
pi_df['match_id']    = pi_df['match_id'].astype(float)

training = training.merge(pi_df, on='match_id', how='left')
training.to_csv(output_path, index=False)

print(f"\nKlaar! {len(training)} rijen.")
print(f"Pi kolommen: {[c for c in training.columns if 'pi' in c]}")


=== Rating bereik ===
Home ratings: min=-1.401, max=1.466, gemiddeld=0.032
Away ratings: min=-1.355, max=1.456, gemiddeld=0.036

=== Top 5 sterkste teams (home) ===
  Manchester City: 1.4663
  Arsenal: 1.4538
  Liverpool: 0.9091
  Chelsea: 0.7925
  Aston Villa: 0.6803

=== Top 5 zwakste teams (home) ===
  Ipswich Town: -0.7752
  Watford: -1.0108
  Southampton: -1.0202
  Sheffield United: -1.2858
  Norwich City: -1.4007

=== Verwacht doelsaldo (pi_expected_gd_pre) ===
count    2665.000000
mean        0.201434
std         0.825909
min        -2.503890
25%        -0.307030
50%         0.195820
75%         0.725320
max         2.891240
Name: pi_expected_gd_pre, dtype: float64

Klaar! 2673 rijen.
Pi kolommen: ['home_pi_home_pre', 'away_pi_away_pre', 'pi_expected_gd_pre']


In [16]:
"""
Team Lineup Strength Overzicht
==============================
Maakt een CSV met per wedstrijd de individuele spelersterktes zodat je
kunt controleren of de berekeningen kloppen.
Kolommen: match_id, round, home_team, away_team, timestamp, status,
          daarna per positie (G, D, M, F) de spelernamen + hun rating
"""

import pandas as pd
import os
import glob

# ── Paden ─────────────────────────────────────────────────────────────────────
PLAYER_DIR    = r"C:\Users\semwi\FPL-Core-Insights\data\Seasonal data\players"
TRAINING_PATH = r"C:\Users\semwi\FPL-Core-Insights\data\training_data.csv"
OUTPUT_PATH   = r"C:\Users\semwi\FPL-Core-Insights\data\player_strength.csv"

# ── Data laden ────────────────────────────────────────────────────────────────
print("📂 Data laden...")
all_files  = sorted(glob.glob(os.path.join(PLAYER_DIR, "*_players.csv")))
players_df = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in all_files],
    ignore_index=True
)
training = pd.read_csv(TRAINING_PATH)

players_df["timestamp"] = pd.to_numeric(players_df["timestamp"], errors="coerce")
training["timestamp"]   = pd.to_numeric(training["timestamp"],   errors="coerce")

# Alleen basiself
starters = players_df[
    players_df["substitute"].astype(str).str.lower() == "false"
].copy()

# ── Historische rating per speler (geen leakage) ──────────────────────────────
starters = starters.sort_values(["player_id", "timestamp"])
starters["rating_historic"] = (
    starters.groupby("player_id")["rating"]
    .transform(lambda x: x.expanding().mean().shift(1))
)
global_avg = starters.groupby("player_id")["rating"].mean().rename("rating_global_avg")
starters   = starters.merge(global_avg, on="player_id", how="left")
starters["rating_best"] = starters["rating_historic"].fillna(starters["rating_global_avg"])

# ── Per wedstrijd + side: spelers als kolommen ───────────────────────────────
print("⚙️  Overzicht bouwen...")

rows = []
for match_id, group in starters.groupby("match_id"):
    # Basisinfo uit training
    match_info = training[training["match_id"] == match_id]
    if match_info.empty:
        continue
    match_info = match_info.iloc[0]

    row = {
        "match_id":   match_id,
        "season":     match_info.get("season", ""),
        "round":      match_info.get("round", ""),
        "timestamp":  match_info.get("timestamp", ""),
        "status":     match_info.get("status", ""),
        "home_team":  match_info.get("home_team", ""),
        "away_team":  match_info.get("away_team", ""),
    }

    for side in ["home", "away"]:
        side_group = group[group["side"] == side].sort_values("position")
        team       = match_info.get(f"{side}_team", side)

        # Gemiddelde sterkte
        avg = side_group["rating_best"].mean()
        row[f"{side}_lineup_strength"] = round(avg, 3) if pd.notna(avg) else None

        # Per positie: spelers + hun rating
        for pos in ["G", "D", "M", "F"]:
            pos_players = side_group[side_group["position"] == pos].reset_index(drop=True)
            for i, (_, p) in enumerate(pos_players.iterrows(), 1):
                name   = p.get("short_name") or p.get("player_name", "")
                rating = round(p["rating_best"], 2) if pd.notna(p.get("rating_best")) else None
                row[f"{side}_{pos}{i}_name"]   = name
                row[f"{side}_{pos}{i}_rating"] = rating

    rows.append(row)

result = pd.DataFrame(rows)

# Sorteer kolommen netjes: info | home spelers | away spelers
info_cols = ["match_id", "season", "round", "timestamp", "status",
             "home_team", "away_team",
             "home_lineup_strength", "away_lineup_strength"]

home_cols = sorted([c for c in result.columns if c.startswith("home_") and c not in info_cols])
away_cols = sorted([c for c in result.columns if c.startswith("away_") and c not in info_cols])

result = result[info_cols + home_cols + away_cols]
result = result.sort_values(["season", "round", "match_id"]).reset_index(drop=True)

result.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ Opgeslagen: {len(result):,} wedstrijden, {len(result.columns)} kolommen")
print(f"   → {OUTPUT_PATH}")
print(f"\nVoorbeeld (laatste 3 wedstrijden):")
print(result[["round","home_team","away_team",
              "home_lineup_strength","away_lineup_strength",
              "home_G1_name","home_G1_rating",
              "away_G1_name","away_G1_rating"]].tail(3).to_string(index=False))

📂 Data laden...
⚙️  Overzicht bouwen...

✅ Opgeslagen: 2,571 wedstrijden, 69 kolommen
   → C:\Users\semwi\FPL-Core-Insights\data\player_strength.csv

Voorbeeld (laatste 3 wedstrijden):
 round        home_team         away_team  home_lineup_strength  away_lineup_strength   home_G1_name  home_G1_rating away_G1_name  away_G1_rating
    38       Luton Town            Fulham                 6.885                 6.950    T. Kaminski            6.94      B. Leno            7.05
    38  Manchester City   West Ham United                 7.396                 7.007      S. Ortega            7.36    A. Aréola            7.11
    38 Sheffield United Tottenham Hotspur                 6.786                 7.072 W. Foderingham            6.99   G. Vicario            7.04
